# 02 — Data Cleaning

### Movie Ticket Booking Analytics · synthetic dataset v2

**Goal:** repair every defect found in `01_data_understanding.ipynb`, prove each
repair with before/after evidence, and produce a file that passes every
validation rule.

**The rule for this notebook:** no row is deleted because a value looks unusual.
A defect is repaired only when the correct value can be recovered from evidence
already in the file — another booking of the same show, the same movie, or the
same customer.

> ⚠️ **All data is synthetic.** Generated by `src/00_generate_data.py` with
> `random.seed(42)`. Movie titles, multiplexes and customers are invented.

The production version of this logic is `src/01_clean_data.py`. This notebook
performs the same steps independently, then checks that both produce an
identical result in the final section.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "bookings_raw.csv").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw" / "bookings_raw.csv"
CLEAN = ROOT / "data" / "clean" / "bookings_clean.csv"
raw_hash_before = hashlib.sha256(RAW.read_bytes()).hexdigest()

START = pd.Timestamp("2025-10-01")
END = pd.Timestamp("2026-09-30")
VALID_TIMES = ("09:00", "12:15", "15:30", "18:45", "21:30", "23:45")
VALID_CITIES = {
    "Mumbai", "Delhi", "Bengaluru", "Hyderabad",
    "Chennai", "Pune", "Kolkata", "Ahmedabad",
}
CITY_MAP = {"Bangalore": "Bengaluru", "Banglore": "Bengaluru"}
PROMO_RATES = {"FIRSTBUY5": 0.05, "WEEKEND10": 0.10,
               "STUDENT15": 0.15, "FAMILY20": 0.20}

# phone must be text: it is an identifier, not a number.
raw = pd.read_csv(RAW, dtype={"phone": "string"})
print("Raw rows :", f"{len(raw):,}")
print("Raw cols :", len(raw.columns))

Raw rows : 10,300
Raw cols : 31


## Step 1 — Deduplicate

A duplicate row is not a second sale. Dropping duplicates first has a second
benefit: we do not waste time repairing the same defect twice.

The table below compares defect counts **before** and **after** deduplication.
The raw counts are the ones quoted in the quality report; the smaller counts are
what cleaning actually has to repair, because a few duplicate copies carried a
defect with them.

In [2]:
before = len(raw)
dup_id = raw.loc[raw["booking_id"].duplicated(keep=False), "booking_id"].iloc[0]

print("Rows before          :", f"{before:,}")
print("Distinct booking_id  :", f"{raw['booking_id'].nunique():,}")
print("Repeated-id rows     :", int(raw["booking_id"].duplicated().sum()))
print()
print("Example: every row for", dup_id)
display(raw[raw["booking_id"] == dup_id])

df = raw.drop_duplicates(subset="booking_id", keep="first").reset_index(drop=True)
print()
print("Rows after           :", f"{len(df):,}")

Rows before          : 10,300
Distinct booking_id  : 10,000
Repeated-id rows     : 300

Example: every row for BK0000095


,booking_id,customer_id,customer_name,customer_email,phone,age,age_group,gender,city,theatre_name,screen,seats_capacity,movie_title,genre,language,rating,format,show_id,show_date,show_time,ticket_type,seats,ticket_price,convenience_fee,promo_code,discount_amount,total_amount,payment_method,booking_channel,booking_status,booking_datetime
94,BK0000095,CUS02609,Myra Tiwari,myra.tiwari2609@example.com,9547587837,45,45-54,Female,hyderabad,Nova Screens - Gachibowli,Screen 5,160,Golconda,Historical,Telugu,8.1,2D,SH000143,2025-10-08,21:30,Premium,4,385,34,NONE,0,1574,NetBanking,Third-Party,Confirmed,2025-10-01 07:11:05
10229,BK0000095,CUS02609,Myra Tiwari,myra.tiwari2609@example.com,9547587837,45,45-54,Female,hyderabad,Nova Screens - Gachibowli,Screen 5,160,Golconda,Historical,Telugu,8.1,2D,SH000143,2025-10-08,21:30,Premium,4,385,34,NONE,0,1574,NetBanking,Third-Party,Confirmed,2025-10-01 07:11:05



Rows after           : 10,000


In [3]:
def defect_counts(frame):
    return {
        "ticket_price <= 0": int((frame["ticket_price"] <= 0).sum()),
        "invalid show_time": int((~frame["show_time"].isin(VALID_TIMES)).sum()),
        "show_date outside period": int(
            (~pd.to_datetime(frame["show_date"], errors="coerce").between(START, END)).sum()
        ),
        "blank customer_email": int(frame["customer_email"].isna().sum()),
        "blank phone": int(frame["phone"].isna().sum()),
        "blank age_group": int(frame["age_group"].isna().sum()),
        "blank promo_code": int(frame["promo_code"].isna().sum()),
        "city label changed": int(
            (frame["city"] != frame["city"].str.strip().str.title().replace(CITY_MAP)).sum()
        ),
    }


comparison = pd.DataFrame({
    "raw file": defect_counts(raw),
    "after dedupe": defect_counts(df),
})
comparison["removed with duplicates"] = comparison["raw file"] - comparison["after dedupe"]
display(comparison)

,raw file,after dedupe,removed with duplicates
ticket_price <= 0,52,50,2
invalid show_time,31,30,1
show_date outside period,20,20,0
blank customer_email,62,60,2
blank phone,61,60,1
blank age_group,51,50,1
blank promo_code,50,50,0
city label changed,242,232,10


**What we see:** 10,300 rows collapse to 10,000. The comparison table shows why
the repair counts later in this notebook are slightly smaller than the raw
counts in the quality report: a duplicated row can carry a defect too, and
removing the copy removes one instance of that defect.

**Treatment:** keep the first occurrence of each `booking_id`. The copies are
exact, so keeping first or last changes nothing.

## Step 2 — Normalise city labels

`groupby("city")` treats `Bengaluru`, `Bangalore`, `Banglore`, `bengaluru` and
`" Bengaluru "` as five different cities. We strip whitespace, apply title case,
then map the two misspellings to the canonical name.

Title case handles `MUMBAI` → `Mumbai`; the explicit map handles spellings that
title case cannot fix.

In [4]:
labels_before = df["city"].nunique()
city_before = df["city"].copy()

df["city"] = df["city"].str.strip().str.title().replace(CITY_MAP)
rows_changed = int((city_before != df["city"]).sum())

print("Distinct labels before:", labels_before)
print("Distinct labels after :", df["city"].nunique())
print("Rows changed          :", rows_changed)
print()
display(df["city"].value_counts().rename("bookings"))

Distinct labels before: 34
Distinct labels after : 8
Rows changed          : 232



city
Mumbai       1889
Delhi        1665
Chennai      1222
Bengaluru    1162
Pune         1151
Hyderabad    1125
Kolkata       941
Ahmedabad     845
Name: bookings, dtype: int64

**What we see:** 34 raw labels become the 8 real cities. The combined Bengaluru
count is what the generator intended as a single city; the variants were only
spelling differences.

**Validation:** the cleaned city must also agree with each customer's other
bookings and with the theatre's city. We check that in the validation step.

## Step 3 — Recover non-positive ticket prices

The data dictionary defines:

`total_amount = ticket_price × seats + convenience_fee − discount_amount`

If the price field is corrupted but the stored total is intact, the price is
recoverable exactly:

`ticket_price = (total_amount − convenience_fee + discount_amount) ÷ seats`

This only works if every other term is valid. We check that the division gives a
whole number before accepting it.

In [5]:
bad_price = df["ticket_price"] <= 0
print("Rows with price <= 0 after dedupe:", int(bad_price.sum()))
print()
display(
    df.loc[bad_price, ["booking_id", "ticket_price", "seats",
                       "convenience_fee", "discount_amount", "total_amount"]].head()
)

recovered = (
    df.loc[bad_price, "total_amount"]
    - df.loc[bad_price, "convenience_fee"]
    + df.loc[bad_price, "discount_amount"]
) / df.loc[bad_price, "seats"]

print()
print("Every recovered price is a whole number:", bool(np.allclose(recovered, recovered.round())))
print("Recovered price range:", int(recovered.min()), "to", int(recovered.max()))

expected_before = (
    df["ticket_price"] * df["seats"]
    + df["convenience_fee"]
    - df["discount_amount"]
)
print("Formula failures before repair:", int((expected_before != df["total_amount"]).sum()))

df.loc[bad_price, "ticket_price"] = recovered.round().astype("int64")

Rows with price <= 0 after dedupe: 50



,booking_id,ticket_price,seats,convenience_fee,discount_amount,total_amount
332,BK0000333,0,2,44,0,734
419,BK0000420,-100,5,41,275,2516
480,BK0000481,-100,4,22,0,1582
582,BK0000583,-100,3,28,0,838
732,BK0000733,-50,2,20,0,800



Every recovered price is a whole number: True
Recovered price range: 210 to 805
Formula failures before repair: 50


In [6]:
expected_after = (
    df["ticket_price"] * df["seats"]
    + df["convenience_fee"]
    - df["discount_amount"]
)
print("Formula failures after repair :", int((expected_after != df["total_amount"]).sum()))
print("Price range after repair      :", int(df["ticket_price"].min()),
      "to", int(df["ticket_price"].max()))
print()
display(
    df.loc[bad_price, ["booking_id", "ticket_price", "seats",
                       "convenience_fee", "discount_amount", "total_amount"]].head()
)

Formula failures after repair : 0
Price range after repair      : 175 to 1510



,booking_id,ticket_price,seats,convenience_fee,discount_amount,total_amount
332,BK0000333,345,2,44,0,734
419,BK0000420,550,5,41,275,2516
480,BK0000481,390,4,22,0,1582
582,BK0000583,270,3,28,0,838
732,BK0000733,390,2,20,0,800


**What we see:** every bad price was recoverable, and the formula failures drop
to zero. The repaired prices are ordinary values in the same range as the rest
of the file (₹175–₹1,510), which is a useful sanity check: a "repaired" price of
₹50,000 would suggest the formula assumption was wrong.

**Why not set the price to the average?** An average would invent revenue that
never existed. The formula recovers the actual amount the customer was charged,
because the stored total never changed.

## Step 4 — Repair show dates and times

`show_date` and `show_time` describe the **show**, not the booking. Every
booking for a `show_id` must agree on them.

When one row is corrupted, the other bookings of the same show still hold the
true value, so the **mode** (most common value) is the repair value. We only
consider values that are already valid: a mode computed over corrupted values
could pick a corrupted value.

In [7]:
df["show_date_dt"] = pd.to_datetime(df["show_date"], errors="coerce")
valid_time = df["show_time"].isin(VALID_TIMES)
valid_date = df["show_date_dt"].between(START, END)

print("Invalid show_time rows   :", int((~valid_time).sum()))
print("Out-of-period date rows  :", int((~valid_date).sum()))
print()

bad_show = df.loc[~valid_time, "show_id"].iloc[0]
print("Before repair, show", bad_show, "has these rows:")
display(df.loc[df["show_id"] == bad_show,
             ["booking_id", "show_id", "show_date", "show_time"]])

Invalid show_time rows   : 30
Out-of-period date rows  : 20

Before repair, show SH000052 has these rows:


,booking_id,show_id,show_date,show_time
24,BK0000025,SH000052,2025-10-10,15:30
175,BK0000176,SH000052,2025-10-10,15:30
193,BK0000194,SH000052,2025-10-10,15:30
196,BK0000197,SH000052,2025-10-10,15:30
239,BK0000240,SH000052,2025-10-10,15:30
240,BK0000241,SH000052,2025-10-10,15:30
274,BK0000275,SH000052,2025-10-10,15:30
276,BK0000277,SH000052,2025-10-10,15:30
277,BK0000278,SH000052,2025-10-10,15:30
279,BK0000280,SH000052,2025-10-10,15:30


In [8]:
time_mode = (
    df.loc[valid_time].groupby("show_id")["show_time"]
    .agg(lambda values: values.mode().iat[0])
)
date_mode = (
    df.loc[valid_date].groupby("show_id")["show_date"]
    .agg(lambda values: values.mode().iat[0])
)

df.loc[~valid_time, "show_time"] = df.loc[~valid_time, "show_id"].map(time_mode)
df.loc[~valid_date, "show_date"] = df.loc[~valid_date, "show_id"].map(date_mode)
df["show_date_dt"] = pd.to_datetime(df["show_date"], errors="coerce")

print("After repair, show", bad_show, "has these rows:")
display(df.loc[df["show_id"] == bad_show,
             ["booking_id", "show_id", "show_date", "show_time"]])
print()
print("Invalid show_time rows  :", int((~df["show_time"].isin(VALID_TIMES)).sum()))
print("Out-of-period date rows :",
      int((~df["show_date_dt"].between(START, END)).sum()))

After repair, show SH000052 has these rows:


,booking_id,show_id,show_date,show_time
24,BK0000025,SH000052,2025-10-10,15:30
175,BK0000176,SH000052,2025-10-10,15:30
193,BK0000194,SH000052,2025-10-10,15:30
196,BK0000197,SH000052,2025-10-10,15:30
239,BK0000240,SH000052,2025-10-10,15:30
240,BK0000241,SH000052,2025-10-10,15:30
274,BK0000275,SH000052,2025-10-10,15:30
276,BK0000277,SH000052,2025-10-10,15:30
277,BK0000278,SH000052,2025-10-10,15:30
279,BK0000280,SH000052,2025-10-10,15:30



Invalid show_time rows  : 0
Out-of-period date rows : 0


**What we see:** the corrupted row now matches the rest of its show. No show had
to be dropped, because every affected show had other valid bookings to copy
from.

**Why the mode is safe here:** a show is one movie at one screen, date and time.
There is exactly one true value, so the majority cannot be wrong unless the
whole show is corrupted — which the understanding notebook showed is not the
case.

## Step 5 — Repair ratings and ages

Both fields are owned by something other than the booking:

- `rating` belongs to the **movie** — every booking of the same title must have
  the same rating.
- `age` belongs to the **customer** — the same `customer_id` cannot be 29 in one
  booking and 150 in another.

So we copy the modal valid value from the same movie or the same customer.

In [9]:
valid_rating = df["rating"].between(0, 10)
print("Out-of-range ratings:", int((~valid_rating).sum()))
display(df.loc[~valid_rating, ["booking_id", "movie_title", "rating"]])

rating_mode = (
    df.loc[valid_rating].groupby("movie_title")["rating"]
    .agg(lambda values: values.mode().iat[0])
)
df.loc[~valid_rating, "rating"] = df.loc[~valid_rating, "movie_title"].map(rating_mode)
df["rating"] = df["rating"].astype(float)

print()
print("Rating range after repair:", round(df["rating"].min(), 1), "to", round(df["rating"].max(), 1))

Out-of-range ratings: 10


,booking_id,movie_title,rating
211,BK0000212,Golconda,-1.0
3985,BK0003986,Mehfil,12.5
4097,BK0004098,Raktham,-1.0
4819,BK0004820,Kanaka,-1.0
4830,BK0004831,Rangmanch,-1.0
5450,BK0005451,Antariksh 2050,-1.0
5798,BK0005799,Dilli Nights,-1.0
5955,BK0005956,Vaanam,-1.0
6483,BK0006484,Aakhri Raasta,-1.0
8139,BK0008140,Chandni Chowk Express,-1.0



Rating range after repair: 6.5 to 8.4


In [10]:
valid_age = df["age"].between(18, 68)
print("Impossible ages:", int((~valid_age).sum()))
display(df.loc[~valid_age, ["booking_id", "customer_id", "age", "age_group"]])

age_mode = (
    df.loc[valid_age].groupby("customer_id")["age"]
    .agg(lambda values: values.mode().iat[0])
)
df.loc[~valid_age, "age"] = df.loc[~valid_age, "customer_id"].map(age_mode)

fallback_age = int(df.loc[valid_age, "age"].median())
fallback_used = int(df["age"].isna().sum())
df["age"] = df["age"].fillna(fallback_age).round().astype("int64")

print()
print("Age fallback used:", fallback_used, "(median age:", fallback_age, ")")
print("Age range after repair:", int(df["age"].min()), "to", int(df["age"].max()))
print()
display(df.loc[~valid_age, ["booking_id", "customer_id", "age", "age_group"]])

Impossible ages: 10


,booking_id,customer_id,age,age_group
2455,BK0002456,CUS03061,-5,45-54
2806,BK0002807,CUS03898,-5,25-34
3078,BK0003079,CUS01190,-5,45-54
3567,BK0003568,CUS00988,-5,25-34
4387,BK0004388,CUS02854,-5,18-24
5992,BK0005993,CUS01808,150,25-34
7049,BK0007050,CUS01982,150,45-54
7281,BK0007282,CUS03503,-5,25-34
7637,BK0007638,CUS02656,-5,45-54
9693,BK0009694,CUS02772,150,25-34



Age fallback used: 0 (median age: 34 )
Age range after repair: 18 to 68



,booking_id,customer_id,age,age_group
2455,BK0002456,CUS03061,48,45-54
2806,BK0002807,CUS03898,30,25-34
3078,BK0003079,CUS01190,50,45-54
3567,BK0003568,CUS00988,27,25-34
4387,BK0004388,CUS02854,18,18-24
5992,BK0005993,CUS01808,26,25-34
7049,BK0007050,CUS01982,50,45-54
7281,BK0007282,CUS03503,32,25-34
7637,BK0007638,CUS02656,51,45-54
9693,BK0009694,CUS02772,30,25-34


**What we see:** the impossible ratings and ages are replaced with the values
that the same movie or customer carries elsewhere. The fallback (median age) was
not needed, because every corrupted age belonged to a repeat customer — exactly
as the generator designed.

**Note:** `age_group` still shows the old band next to the repaired age. We
re-derive it in Step 7, so it cannot disagree with `age`.

## Step 6 — Fill missing contact details

A blank email or phone is not the text "Unknown". It means the field was not
recorded for this booking. The same customer's other bookings usually carry the
value, so we copy it.

If a customer had only one booking, there is no evidence to copy. In that case
we use a deterministic placeholder and log it — we do **not** invent a plausible
address.

In [11]:
for column, placeholder in [("customer_email", None), ("phone", "0000000000")]:
    missing = df[column].isna() | (df[column].astype("string").str.strip() == "")
    print(f"{column}: {int(missing.sum())} blanks")

    fill_values = df.loc[~missing].groupby("customer_id")[column].first()
    df.loc[missing, column] = df.loc[missing, "customer_id"].map(fill_values)

    still_missing = df[column].isna() | (df[column].astype("string").str.strip() == "")
    if still_missing.any():
        if column == "customer_email":
            df.loc[still_missing, column] = (
                df.loc[still_missing, "customer_id"].str.lower() + "@example.com"
            )
        else:
            df.loc[still_missing, column] = placeholder

    df[column] = df[column].astype("string").str.strip()
    print(f"  recovered from another booking: {int(missing.sum()) - int(still_missing.sum())}")
    print(f"  placeholders used             : {int(still_missing.sum())}")
    print(f"  blanks remaining              : {int(df[column].isna().sum())}")

customer_email: 60 blanks
  recovered from another booking: 60
  placeholders used             : 0
  blanks remaining              : 0
phone: 60 blanks
  recovered from another booking: 60
  placeholders used             : 0
  blanks remaining              : 0


**What we see:** every blank was recoverable from another booking of the same
customer, so no placeholder was needed. That is consistent with the generator,
which only blanked contact details on repeat customers.

**Why this matters:** a blank email breaks the link between a booking and a
customer in any downstream join. Filling it from the same customer preserves
that link without inventing a new identity.

## Step 7 — Re-derive `age_group`

`age_group` is a function of `age`, so it should never be stored independently
without being re-derived after an age repair. We overwrite it for every row and
check whether any previously filled value disagreed.

In [12]:
def age_band(age):
    if age < 25:
        return "18-24"
    if age < 35:
        return "25-34"
    if age < 45:
        return "35-44"
    if age < 55:
        return "45-54"
    return "55+"


age_group_before = df["age_group"].copy()
df["age_group"] = df["age"].map(age_band)

print("Blanks before        :", int(age_group_before.isna().sum()))
print("Rows that disagreed  :", int(
    (age_group_before.notna() & (age_group_before != df["age_group"])).sum()
))
print("Bands after          :", sorted(df["age_group"].unique()))
print()
display(df["age_group"].value_counts().sort_index().rename("bookings"))

Blanks before        : 50
Rows that disagreed  : 0
Bands after          : ['18-24', '25-34', '35-44', '45-54', '55+']



age_group
18-24    2180
25-34    3020
35-44    2370
45-54    1607
55+       823
Name: bookings, dtype: int64

**What we see:** 50 blanks were filled, and zero previously filled values
disagreed with the age. That is a good sign for the source data: `age_group` was
always consistent when present, and only the blank rows needed work.

If any row had disagreed, the re-derived value would still win, because `age` is
the source of truth and `age_group` is only a convenience label.

## Step 8 — Infer missing promo codes

A blank `promo_code` can mean two different things:

1. No promo was used → discount is 0 → the correct value is `NONE`.
2. A promo was used but not recorded → discount is greater than 0.

In case 2 the discount rate identifies the promo, because the four rates are
unique:

| Code | Rate |
|---|---|
| FIRSTBUY5 | 5% |
| WEEKEND10 | 10% |
| STUDENT15 | 15% |
| FAMILY20 | 20% |

We compute `discount ÷ (ticket_price × seats)` and pick the nearest rate. This
is why Step 3 had to repair prices first — the rate calculation needs a valid
price.

In [13]:
blank_promo = df["promo_code"].isna() | (df["promo_code"].astype("string").str.strip() == "")
print("Blank promo codes:", int(blank_promo.sum()))
print()
display(pd.DataFrame({
    "promo_code": list(PROMO_RATES),
    "discount_rate": [f"{rate:.0%}" for rate in PROMO_RATES.values()],
}))

subtotal = (df["ticket_price"] * df["seats"]).replace(0, np.nan)
discount_rate = df["discount_amount"] / subtotal

inferred = []
unmatched = 0
for index in df.index[blank_promo]:
    if df.at[index, "discount_amount"] == 0:
        inferred.append("NONE")
        continue
    rate = discount_rate.at[index]
    if pd.isna(rate):
        inferred.append("NONE")
        unmatched += 1
        continue
    code = min(PROMO_RATES, key=lambda key: abs(PROMO_RATES[key] - rate))
    if abs(PROMO_RATES[code] - rate) > 0.01:
        code = "NONE"
        unmatched += 1
    inferred.append(code)

df.loc[blank_promo, "promo_code"] = inferred
df["promo_code"] = df["promo_code"].astype("string").str.strip()

print()
print("Matched to a rate:", int(blank_promo.sum()) - unmatched)
print("Set to NONE      :", int((pd.Series(inferred) == "NONE").sum()))
print("Unmatched        :", unmatched)
print()
display(df.loc[blank_promo, ["booking_id", "ticket_price", "seats",
                              "discount_amount", "promo_code"]].head(8))

Blank promo codes: 50



,promo_code,discount_rate
0,FIRSTBUY5,5%
1,WEEKEND10,10%
2,STUDENT15,15%
3,FAMILY20,20%



Matched to a rate: 50
Set to NONE      : 35
Unmatched        : 0



,booking_id,ticket_price,seats,discount_amount,promo_code
17,BK0000018,230,2,69,STUDENT15
22,BK0000023,390,1,0,NONE
73,BK0000074,235,2,70,STUDENT15
195,BK0000196,605,2,0,NONE
280,BK0000281,330,1,66,FAMILY20
534,BK0000535,405,3,0,NONE
716,BK0000717,555,4,0,NONE
902,BK0000903,710,4,0,NONE


In [14]:
print("Discount without a promo:",
      int(((df["discount_amount"] > 0) & (df["promo_code"] == "NONE")).sum()))
print("Blanks remaining       :", int(df["promo_code"].isna().sum()))

Discount without a promo: 0
Blanks remaining       : 0


**What we see:** of the blank promo codes, 15 had a real discount and were mapped
to the matching rate; the rest had zero discount and became `NONE`. No row now
has a discount without a promo code.

**Why not fill every blank with `NONE`?** That would erase 15 genuine discounts
and make the promo analysis understate usage. Why not fill every blank with the
most common promo? That would invent discounts that never happened.

**Remaining uncertainty:** if a discount were corrupted as well, the rate would
not match and the code would fall back to `NONE` with a warning. None did.

## Step 9 — Derive reporting fields

Two fields are calculated, never stored raw:

- `customer_type`: New = 1 booking, Returning = 2–3, Loyal = 4+ in the window.
- `show_occupancy_rate`: confirmed seats for the show ÷ `seats_capacity`.

Occupancy is computed **per show**, not per booking, and cancelled bookings are
excluded because those seats were released.

In [15]:
bookings_per_customer = df.groupby("customer_id").size()
customer_bookings = df["customer_id"].map(bookings_per_customer)
df["customer_type"] = np.select(
    [customer_bookings == 1, customer_bookings <= 3],
    ["New", "Returning"],
    default="Loyal",
)

segment = pd.DataFrame({
    "customers": df.drop_duplicates("customer_id")["customer_type"].value_counts(),
    "bookings": df["customer_type"].value_counts(),
}).reindex(["New", "Returning", "Loyal"])

segment["customer_share_%"] = (segment["customers"] / segment["customers"].sum() * 100).round(1)
segment["booking_share_%"] = (segment["bookings"] / segment["bookings"].sum() * 100).round(1)
display(segment)

,customers,bookings,customer_share_%,booking_share_%
customer_type,,,,
New,1090,1090,31.3,10.9
Returning,1507,3581,43.3,35.8
Loyal,883,5329,25.4,53.3


In [16]:
confirmed = df[df["booking_status"] == "Confirmed"]
seats_sold = confirmed.groupby("show_id")["seats"].sum()
capacity = df.groupby("show_id")["seats_capacity"].first()
occupancy = seats_sold.reindex(capacity.index, fill_value=0) / capacity

df["show_occupancy_rate"] = df["show_id"].map(occupancy).round(6)

print("Shows            :", len(occupancy))
print("Occupancy min    :", round(occupancy.min(), 3))
print("Occupancy mean   :", round(occupancy.mean(), 3))
print("Occupancy max    :", round(occupancy.max(), 3))
print("Shows above 100% :", int((occupancy > 1).sum()))

Shows            : 290
Occupancy min    : 0.05
Occupancy mean   : 0.402
Occupancy max    : 0.915
Shows above 100% : 0


**What we see:** Loyal customers are 25.4% of customers but 53.3% of bookings.
Occupancy is 5.0%–91.5%, mean 40.2%, and no show exceeds 100%.

The occupancy value is identical for every booking of the same show, because it
describes the show. If you group by `show_id` and take the first value, you get
the show-level table; if you average the booking-level column without grouping,
you weight busy shows more heavily. The validation section checks the show-level
version, which is the one the report quotes.

## Step 10 — Full validation

These are the same rules as `src/01_clean_data.py`. The table is built from the
data, so a failure would appear here rather than being hidden in prose.

In [17]:
required_columns = [
    "booking_id", "customer_id", "customer_name", "customer_email", "phone",
    "age", "age_group", "gender", "city", "theatre_name", "screen",
    "seats_capacity", "movie_title", "genre", "language", "rating", "format",
    "show_id", "show_date", "show_time", "ticket_type", "seats",
    "ticket_price", "convenience_fee", "promo_code", "discount_amount",
    "total_amount", "payment_method", "booking_channel", "booking_status",
    "booking_datetime", "customer_type", "show_occupancy_rate",
]

blank_total = 0
for column in required_columns:
    series = df[column]
    blanks = int(series.isna().sum())
    if str(series.dtype) in ("object", "string", "str"):
        blanks += int((series.astype("string").str.strip() == "").sum())
    blank_total += blanks

show_conflicts = int(
    (df.groupby("show_id")[
        ["movie_title", "theatre_name", "screen", "seats_capacity",
         "show_date", "show_time"]
    ].nunique() > 1).any(axis=1).sum()
)
customer_conflicts = int(
    (df.groupby("customer_id")[["customer_name", "gender", "city"]]
     .nunique() > 1).any(axis=1).sum()
)

show_datetime = pd.to_datetime(df["show_date"] + " " + df["show_time"], errors="coerce")
booking_datetime = pd.to_datetime(df["booking_datetime"], errors="coerce")
booked_after_show = int((booking_datetime > show_datetime).sum())

phone_valid = df["phone"].str.fullmatch(r"[6-9]\d{9}").fillna(False)

expected = (
    df["ticket_price"] * df["seats"]
    + df["convenience_fee"]
    - df["discount_amount"]
)

checks = pd.DataFrame([
    ("Rows = 10,000", len(df) == 10000, f"{len(df):,}"),
    ("Unique booking_id = 10,000", df["booking_id"].nunique() == 10000,
     f"{df['booking_id'].nunique():,}"),
    ("No blanks in required fields", blank_total == 0, f"{blank_total}"),
    ("Revenue formula holds", int((expected != df["total_amount"]).sum()) == 0,
     f"{int((expected != df['total_amount']).sum())} failures"),
    ("Ticket price > 0", int((df["ticket_price"] <= 0).sum()) == 0,
     f"min {int(df['ticket_price'].min())}"),
    ("Show time in valid slots", int((~df["show_time"].isin(VALID_TIMES)).sum()) == 0,
     f"{int((~df['show_time'].isin(VALID_TIMES)).sum())} invalid"),
    ("Show date inside window", int((~df["show_date_dt"].between(START, END)).sum()) == 0,
     f"{df['show_date'].min()} to {df['show_date'].max()}"),
    ("Rating 0-10", int((~df["rating"].between(0, 10)).sum()) == 0,
     f"{df['rating'].min():.1f} to {df['rating'].max():.1f}"),
    ("Age 18-68", int((~df["age"].between(18, 68)).sum()) == 0,
     f"{int(df['age'].min())} to {int(df['age'].max())}"),
    ("Exactly 8 canonical cities", set(df["city"]) == VALID_CITIES,
     f"{df['city'].nunique()} labels"),
    ("Show consistency", show_conflicts == 0, f"{show_conflicts} conflicts"),
    ("Customer consistency", customer_conflicts == 0, f"{customer_conflicts} conflicts"),
    ("Occupancy within 0-1", int((~df["show_occupancy_rate"].between(0, 1)).sum()) == 0,
     f"{df['show_occupancy_rate'].min():.3f} to {df['show_occupancy_rate'].max():.3f}"),
    ("Confirmed revenue = INR 10,787,567",
     int(df.loc[df["booking_status"] == "Confirmed", "total_amount"].sum()) == 10787567,
     f"{int(df.loc[df['booking_status'] == 'Confirmed', 'total_amount'].sum()):,}"),
    ("No discount without a promo",
     int(((df["discount_amount"] > 0) & (df["promo_code"] == "NONE")).sum()) == 0,
     f"{int(((df['discount_amount'] > 0) & (df['promo_code'] == 'NONE')).sum())}"),
    ("Seats <= capacity", int((df["seats"] > df["seats_capacity"]).sum()) == 0,
     f"{int((df['seats'] > df['seats_capacity']).sum())}"),
    ("Booking before show start", booked_after_show == 0, f"{booked_after_show}"),
    ("Phone is 10 digits starting 6-9", int((~phone_valid).sum()) == 0,
     f"{int((~phone_valid).sum())} invalid"),
    ("Raw file untouched",
     hashlib.sha256(RAW.read_bytes()).hexdigest() == raw_hash_before,
     "SHA256 match"),
], columns=["check", "passed", "detail"])

display(checks)
print("Checks passed:", int(checks["passed"].sum()), "of", len(checks))

,check,passed,detail
0,"Rows = 10,000",True,"10,000"
1,"Unique booking_id = 10,000",True,"10,000"
2,No blanks in required fields,True,0
3,Revenue formula holds,True,0 failures
4,Ticket price > 0,True,min 175
5,Show time in valid slots,True,0 invalid
6,Show date inside window,True,2025-10-02 to 2026-09-27
7,Rating 0-10,True,6.5 to 8.4
8,Age 18-68,True,18 to 68
9,Exactly 8 canonical cities,True,8 labels


Checks passed: 19 of 19


**What we see:** all checks pass. Two deserve a note:

- **"Booking before show start"** could only be tested after the date/time
  repair. A corrupted time would have produced false failures.
- **"Confirmed revenue"** is the reconciliation anchor. It must match the
  generator's clean base (₹10,787,567), not the duplicate-inflated raw total
  (₹11,110,984).

The last check compares the raw file's SHA256 from the start of the notebook
with its hash now. A mismatch would mean the notebook modified the evidence it
is supposed to preserve.

## Step 11 — Compare with the production script

The notebook and `src/01_clean_data.py` implement the same rules independently.
If they agree on every value, the logic is reproducible and not an artefact of
one implementation.

`assert_frame_equal` is the strictest comparison pandas offers. We disable dtype
checking only because CSV round-tripping can turn a `str` column into `object`
without changing a single value.

In [18]:
mine = df.drop(columns=["show_date_dt"]).sort_values("booking_id").reset_index(drop=True)
script = pd.read_csv(CLEAN, dtype={"phone": "string"}).sort_values("booking_id").reset_index(drop=True)

print("Notebook shape:", mine.shape)
print("Script shape  :", script.shape)

try:
    pd.testing.assert_frame_equal(
        mine, script, check_dtype=False, check_exact=False, rtol=1e-9, atol=1e-9
    )
    print()
    print("MATCH: notebook and script produce identical data.")
except AssertionError as error:
    print()
    print("MISMATCH:")
    print(str(error)[:2000])

Notebook shape: (10000, 33)
Script shape  : (10000, 33)

MATCH: notebook and script produce identical data.


**What we see:** the two implementations agree on all 10,000 rows and 33
columns. That means the cleaning logic is deterministic: re-running either path
produces the same file.

The script also writes `reports/cleaning_log.txt`, which records the before/after
count for every step. The log is the audit trail; this notebook is the
explanation.

## Practice tasks

1. **Repair order.** What would go wrong if we inferred promo codes before
   repairing prices? Try computing `discount ÷ (ticket_price × seats)` on the
   original bad-price rows.
2. **Mode vs mean.** Why is the mode the right repair for `show_time`, but the
   median the right fallback for `age`? What would a mean show time even mean?
3. **Duplicates and defects.** Pick one defect category from the comparison
   table where the raw count is higher than the post-dedupe count. Find the
   duplicate row responsible and explain why repairing before deduplicating
   would waste work.
4. **Occupancy weighting.** Compute mean occupancy two ways: average the
   show-level values, then average the booking-level column. Explain the gap.
5. **Placeholder risk.** Change the email fallback to `unknown@example.com` for
   all unrecoverable rows. What analysis would break, and why is a
   customer-specific placeholder safer?

Write your answers in a new markdown cell below this one. Do not modify the raw
CSV.